In [ ]:
# Cell 1: Setup & Core Dependencies
import getpass
from typing import Annotated, Sequence, TypedDict, Literal
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Secure API Key Entry for the demonstration
anthropic_api_key = getpass.getpass("Enter your Anthropic API Key: ")

# Initialize the Model 
# Using a temperature of 0 ensures strict, predictable routing from the supervisor
llm = ChatAnthropic(
    model="claude-sonnet-4-6", 
    temperature=0, 
    anthropic_api_key=anthropic_api_key
)
print("✅ Environment initialized.")

In [ ]:
# Cell 2: Graph State & Supervisor Node
# Define our worker fleet
members = ["Researcher", "Planner", "Writer"]
options = ["FINISH"] + members

system_prompt = (
    "You are a lead technical supervisor managing a workflow between these agents: {members}. "
    "Given the user request and the conversation history, respond with the name of the worker to act next. "
    "Each worker performs a specialized task and returns their results. "
    "When all requirements of the user's prompt are fully satisfied, respond with FINISH."
)

class SupervisorState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next: str

def supervisor_node(state: SupervisorState) -> dict:
    print("\n🧠 [NODE -> SUPERVISOR]: Evaluating system state and routing next task...")
    messages = [
        {"role": "system", "content": system_prompt.format(members=", ".join(members))}
    ] + list(state["messages"])
    
    # We use structured output to force the LLM to pick exactly one valid option
    response = llm.with_structured_output(
        {"name": "route", "description": "Select the next role.", 
         "parameters": {
             "title": "routeSchema", 
             "type": "object", 
             "properties": {"next": {"type": "string", "enum": options}},
             "required": ["next"]
         }}
    ).invoke(messages)
    
    return {"next": response["next"]}